## 🔴 Bronze Layer - Raw Data Ingestion
This notebook ingests the raw Chicago Crime CSV file from our Unity Catalog Volume and saves it as a Delta table without any transformations. This is our raw data vault.

In [0]:
# Load raw CSV from Unity Catalog Volume
file_path = "/Volumes/workspace/default/crime_data/Crimes_-_2021_to_Present_20260520.csv"

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

print(f"Total Records: {df_raw.count()}")
df_raw.printSchema()

Total Records: 1286734
root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



### 👀 Preview Raw Data
Quick look at the first 10 rows to understand what columns we have before any cleaning.

In [0]:
display(df_raw.limit(10))

ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
14193417,JK249977,05/11/2026 12:00:00 AM,058XX N ROGERS AVE,0710,THEFT,THEFT FROM MOTOR VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),false,false,1711,17,39,13,06,1146681,1938590,2026,05/18/2026 03:42:59 PM,41.987468468,-87.735867509,"(41.987468468, -87.735867509)"
14193533,JK250272,05/11/2026 12:00:00 AM,085XX S HERMITAGE AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,false,false,614,6,18,71,06,1166163,1848161,2026,05/18/2026 03:42:59 PM,41.738928101,-87.666794079,"(41.738928101, -87.666794079)"
14193357,JK249936,05/11/2026 12:00:00 AM,057XX W CHICAGO AVE,0610,BURGLARY,FORCIBLE ENTRY,COMMERCIAL / BUSINESS OFFICE,false,false,1511,15,29,25,05,1137755,1904757,2026,05/18/2026 03:42:59 PM,41.894793399,-87.769516687,"(41.894793399, -87.769516687)"
14197599,JK254317,05/11/2026 12:00:00 AM,028XX N BURLING ST,0820,THEFT,$500 AND UNDER,APARTMENT,false,false,1934,19,44,6,06,1170810,1919094,2026,05/18/2026 03:42:59 PM,41.933474653,-87.647693928,"(41.933474653, -87.647693928)"
14194155,JK250986,05/11/2026 12:00:00 AM,006XX E 112TH ST,0890,THEFT,FROM BUILDING,RESIDENCE - YARD (FRONT / BACK),false,false,531,5,9,50,06,1182739,1830829,2026,05/18/2026 03:42:59 PM,41.690998601,-87.6065992,"(41.690998601, -87.6065992)"
14193250,JK249887,05/11/2026 12:00:00 AM,034XX N HALSTED ST,0820,THEFT,$500 AND UNDER,BAR OR TAVERN,false,false,1924,19,44,6,06,1170308,1923273,2026,05/18/2026 03:42:59 PM,41.944953005,-87.649416212,"(41.944953005, -87.649416212)"
14193327,JK249855,05/11/2026 12:00:00 AM,080XX S HALSTED ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,STREET,false,true,621,6,17,71,08B,1172364,1851615,2026,05/18/2026 03:42:59 PM,41.748272295,-87.643973455,"(41.748272295, -87.643973455)"
14195794,JK253103,05/11/2026 12:00:00 AM,022XX N MILWAUKEE AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,OTHER (SPECIFY),false,false,1431,14,1,22,11,1158133,1914486,2026,05/18/2026 03:42:59 PM,41.921098715,-87.694407322,"(41.921098715, -87.694407322)"
14193271,JK249848,05/11/2026 12:00:00 AM,082XX S ELIZABETH ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,STREET,false,true,613,6,17,71,08B,1169426,1850084,2026,05/18/2026 03:42:59 PM,41.744135109,-87.654783514,"(41.744135109, -87.654783514)"
14194194,JK250679,05/11/2026 12:00:00 AM,007XX W Bittersweet Pl,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,false,false,1915,19,46,3,14,1170531,1927183,2026,05/18/2026 03:42:59 PM,41.955677299,-87.64848172,"(41.955677299, -87.64848172)"


### 💾 Save to Bronze Delta Table
Writing raw data as-is into a Delta table. No transformations applied — this preserves the original data exactly as received.

In [0]:
# Rename columns to remove spaces (Delta Lake requirement)
df_clean_cols = df_raw \
    .withColumnRenamed("Case Number", "Case_Number") \
    .withColumnRenamed("Primary Type", "Primary_Type") \
    .withColumnRenamed("Location Description", "Location_Description") \
    .withColumnRenamed("Community Area", "Community_Area") \
    .withColumnRenamed("FBI Code", "FBI_Code") \
    .withColumnRenamed("X Coordinate", "X_Coordinate") \
    .withColumnRenamed("Y Coordinate", "Y_Coordinate") \
    .withColumnRenamed("Updated On", "Updated_On")

# Save as Bronze Delta Table
df_clean_cols.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_crimes")

print("✅ Bronze Delta table created successfully!")

✅ Bronze Delta table created successfully!


### ✅ Verify Bronze Table
Confirm the data landed correctly by checking the total record count in our new Delta table.

In [0]:
df_check = spark.sql("SELECT COUNT(*) as total_records FROM workspace.default.bronze_crimes")
display(df_check)

total_records
1286734
